# GLOSSATE — Output Formats Test Drive

Hands-on notebook for **every output GLOSSATE can produce**: SRT captions, a burned-in video, and the four Markdown "notes" shapes (with timestamps on/off). Run the cells top to bottom.

**Before you start**
1. **Runtime → Change runtime type → GPU** (A100 ideal; any CUDA GPU works).
2. In your Google Drive, create **`GLOSSATE/input/`** and drop a **video** in it. Outputs land in **`GLOSSATE/output/`**.
3. Gemma weights are gated — accept the license on the model page and log in (cell 2).

The notebook opens **one shared `Session`** so the models load once and the transcript is cached: the first test does the heavy ASR + model load, every later cell is fast.

## 1. Install GLOSSATE (CUDA)

In [ ]:
GLOSSATE_INSTALL = "glossate"   # or "git+https://github.com/anaxoniclabs/GLOSSATE.git@main"

!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q --upgrade pip
!python -m pip install -q "{GLOSSATE_INSTALL}[cuda,detect]"
# Gemma 4 is recent — keep transformers/accelerate current.
!python -m pip install -q --upgrade "transformers>=5.5.0" accelerate

## 2. Hugging Face login (gated Gemma weights)

Markdown notes are produced by Gemma, so you need access. Accept the license on e.g. [`google/gemma-4-E4B-it`](https://huggingface.co/google/gemma-4-E4B-it), then paste a token from <https://huggingface.co/settings/tokens>.

In [ ]:
from huggingface_hub import login
login()  # interactive; or login(token="hf_...")

## 3. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No CUDA. Runtime > Change runtime type > GPU."
print("CUDA:", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))
!nvidia-smi

## 4. Mount Drive and find your input

Looks in **`GLOSSATE/input/`** and auto-picks the first media file. To force a specific file, set `INPUT_NAME = "myclip.mp4"`.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

INPUT_DIR  = Path("/content/drive/MyDrive/GLOSSATE/input")
OUTPUT_DIR = Path("/content/drive/MyDrive/GLOSSATE/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_NAME = None  # e.g. "myclip.mp4"; None = auto-pick first file found

VIDEO_EXT = {".mp4", ".mov", ".mkv", ".webm", ".m4v", ".avi"}
AUDIO_EXT = {".wav", ".mp3", ".m4a", ".flac"}

assert INPUT_DIR.exists(), f"Missing {INPUT_DIR} — create GLOSSATE/input in Drive and add a video."
media = sorted(p for p in INPUT_DIR.iterdir() if p.suffix.lower() in (VIDEO_EXT | AUDIO_EXT))
print("Found in GLOSSATE/input:")
for p in media: print("  -", p.name)
assert media, "No media files in GLOSSATE/input."

INPUT_PATH = (INPUT_DIR / INPUT_NAME) if INPUT_NAME else media[0]
assert INPUT_PATH.exists(), f"Not found: {INPUT_PATH}"
IS_VIDEO = INPUT_PATH.suffix.lower() in VIDEO_EXT
STEM = INPUT_PATH.stem
print("\nUsing:", INPUT_PATH.name, "(video)" if IS_VIDEO else "(audio)")

## 5. Settings

In [ ]:
# Languages
SOURCE_LANG = None     # None = auto-detect; or an ISO code: "en", "ar", "tr"...
TARGET_LANG = "en"     # translate INTO this for the Level-2 tests

# Models (CUDA). Markdown notes need a Gemma / chat model (not NLLB).
ASR_MODEL = "turbo"        # tiny|base|small|medium|large|turbo
MT_MODEL  = "gemma-4-e4b"  # gemma-4-e2b|e4b|26b|31b

# Backends — leave as-is for a Colab GPU.
DEVICE, ASR_BACKEND, MT_BACKEND, COMPUTE_TYPE = "cuda", "faster-whisper", "transformers", "auto"
print("Translating into:", TARGET_LANG)

## 6. Display helpers

In [ ]:
import glossate
from IPython.display import Markdown, display

def out_path(label, ext):
    """A path inside Drive's GLOSSATE/output."""
    return str(OUTPUT_DIR / f"{STEM}.{label}.{ext}")

def show_srt(path, lines=40):
    text = Path(path).read_text(encoding="utf-8")
    print(f"--- {path} ---\n" + "\n".join(text.splitlines()[:lines]))

def show_md(path, limit=4000):
    text = Path(path).read_text(encoding="utf-8")
    print(f"--- {path}  ({len(text)} chars) ---")
    display(Markdown(text[:limit] + ("\n\n…(truncated)…" if len(text) > limit else "")))

print("glossate", glossate.__version__, "ready.")

## 7. Open the shared session

Models load **once** here and stay in VRAM until the cleanup cell at the very bottom. Run that cleanup cell when you're finished to free the GPU.

In [ ]:
S = glossate.Session(
    asr_model=ASR_MODEL, mt_model=MT_MODEL, device=DEVICE,
    asr_backend=ASR_BACKEND, mt_backend=MT_BACKEND, compute_type=COMPUTE_TYPE,
)
S.__enter__()
print("Session open.")

### Peek at the transcript

The first call runs ASR (slow) and caches it — every later cell reuses it. Note that after translation each cue keeps its original in `source_text`/`source_lang`, which is what lets one run emit bilingual notes.

In [ ]:
cues = S.transcribe(INPUT_PATH, source=SOURCE_LANG)   # cached after this
print(f"{len(cues)} cues | detected language: {cues[0].lang if cues else '?'}")
for c in cues[:3]:
    print(f"  [{c.start:6.2f}-{c.end:6.2f}] {c.text!r}")

## 8. SRT — transcription only (Level 1)

No `--target`, so cues stay in the source language. Plain timed captions; good as accessibility subtitles.

In [ ]:
p = S.subtitle(INPUT_PATH, source=SOURCE_LANG, format="srt",
               output=out_path("src", "srt"))
show_srt(p)

## 9. SRT — transcribe + translate (Level 2)

With `target=...`, Gemma translates each cue **in place**; timestamps are preserved.

In [ ]:
p = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="srt",
               output=out_path(TARGET_LANG, "srt"))
show_srt(p)

## 10. SRT — translate + burn into the video

`subtitle_video` writes the SRT and then hard-burns it into a new MP4. Video input only.

In [ ]:
if IS_VIDEO:
    video_out = str(OUTPUT_DIR / f"{STEM}.{TARGET_LANG}.subbed.mp4")
    burned = S.subtitle_video(
        INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="srt",
        subtitle_output=out_path(f"{TARGET_LANG}.burn", "srt"), output=video_out,
    )
    print("Burned video:", burned)
else:
    print("Input is audio — skipping burn (needs a video).")

## 11. Markdown — Level 1 notes (original language)

Here's where Gemma earns its keep: it **reflows** the choppy transcript into clean, readable paragraphs (it does *not* translate). Each cell below writes **two** files — timestamps on and off — so you can see the difference. Timestamps are sparse `[m:ss]` markers placed by code (never by the model).

In [ ]:
on  = S.subtitle(INPUT_PATH, source=SOURCE_LANG, format="md",
                 md_timestamps=True,  output=out_path("notes.src.ts", "md"))
off = S.subtitle(INPUT_PATH, source=SOURCE_LANG, format="md",
                 md_timestamps=False, output=out_path("notes.src", "md"))
show_md(on); show_md(off)

## 12. Markdown — Level 2, translated only (`md_scope="translated"`)

Same readable prose, but in the target language.

In [ ]:
on  = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="md",
                 md_scope="translated", md_timestamps=True,
                 output=out_path(f"notes.{TARGET_LANG}.ts", "md"))
off = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="md",
                 md_scope="translated", md_timestamps=False,
                 output=out_path(f"notes.{TARGET_LANG}", "md"))
show_md(on); show_md(off)

## 13. Markdown — Level 2, bilingual **two-prose** (`md_scope="both"`)

Two sections: the original-language prose, then the translated prose. Best for reading the whole thing in either language.

In [ ]:
on  = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="md",
                 md_scope="both", md_layout="two-prose", md_timestamps=True,
                 output=out_path("notes.both.twoprose.ts", "md"))
off = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="md",
                 md_scope="both", md_layout="two-prose", md_timestamps=False,
                 output=out_path("notes.both.twoprose", "md"))
show_md(on); show_md(off)

## 14. Markdown — Level 2, bilingual **dual-stack** (sentence pairs)

For language study: each source **sentence** is followed by its translation. Sentences are rebuilt from the fragments and translated at sentence granularity (not per-cue), so the pairs line up grammatically.

In [ ]:
on  = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="md",
                 md_scope="both", md_layout="dual-stack", md_timestamps=True,
                 output=out_path("notes.both.dualstack.ts", "md"))
off = S.subtitle(INPUT_PATH, source=SOURCE_LANG, target=TARGET_LANG, format="md",
                 md_scope="both", md_layout="dual-stack", md_timestamps=False,
                 output=out_path("notes.both.dualstack", "md"))
show_md(on); show_md(off)

## 15. Close the session

Frees the GPU. Re-run cell 7 to reopen if you want to experiment more.

In [ ]:
S.__exit__(None, None, None)
print("Session closed, GPU memory released.")

## 16. The same things from the CLI (reference)

Every option above has a CLI flag. **Note:** each `glossate` run loads the model fresh (no shared session), so it's slower than the cells above — use it to learn the flags. Colab substitutes `{VAR}` into `!` commands.

In [ ]:
# Transcribe-only SRT:
!glossate "{INPUT_PATH}" --format srt -o "{OUTPUT_DIR}/{STEM}.cli.srt" --device cuda

# Translate + bilingual dual-stack notes, no timestamps (uncomment to run):
# !glossate "{INPUT_PATH}" --target {TARGET_LANG} --format md \
#     --md-scope both --md-layout dual-stack --no-md-timestamps \
#     -o "{OUTPUT_DIR}/{STEM}.cli.dualstack.md" --device cuda

# Translate + burn:
# !glossate "{INPUT_PATH}" --target {TARGET_LANG} --burn \
#     --burn-output "{OUTPUT_DIR}/{STEM}.cli.subbed.mp4" --device cuda

## 17. Everything written to Drive

In [ ]:
print("Outputs in", OUTPUT_DIR, ":")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:42} {p.stat().st_size:>11,} bytes")